# 🔒 PrivAI — Training YOLOv8 / YOLO11 Lokal

Notebook ini memandu kamu melatih model deteksi dokumen pribadi menggunakan **YOLO11n** (atau YOLOv8) di GPU NVIDIA lokal.

---

## 📋 Alur Notebook

| Step | Sel | Keterangan |
|------|-----|------------|
| 1 | Instalasi | Install library yang dibutuhkan |
| 2 | Cek GPU | Pastikan CUDA tersedia |
| 3 | Konfigurasi | Atur path dataset & hyperparameter |
| 4 | Auto Batch | Hitung batch size sesuai VRAM |
| 5 | Training | Jalankan training |
| 6 | Validasi | Evaluasi mAP model hasil training |
| 7 | Inferensi | Coba model pada gambar baru |

---

> **💡 Tips:** Jalankan sel dari atas ke bawah secara berurutan. Klik sel → `Shift+Enter` untuk menjalankan.

## Step 1 — Instalasi Library

Install `ultralytics` (berisi YOLO) dan `PyTorch` versi CUDA.

> ⚠️ **Penting:** Jika kamu menggunakan **Google Colab**, PyTorch CUDA sudah ter-install otomatis. Jika di lokal, pastikan install versi yang sesuai dengan driver NVIDIA-mu di [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/).

```
# Versi CUDA yang umum:
# CUDA 11.8 → pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# CUDA 12.1 → pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

In [12]:
# Install ultralytics (sudah include semua dependency YOLO)
# Uncomment baris di bawah jika belum ter-install

# !pip install ultralytics

# ── Untuk Google Colab / environment fresh ──
# !pip install ultralytics torch torchvision torchaudio

# Verifikasi instalasi
import ultralytics
print(f"✅ Ultralytics version: {ultralytics.__version__}")

✅ Ultralytics version: 8.4.25


## Step 2 — Cek GPU & CUDA

Sebelum training, pastikan PyTorch mengenali GPU-mu.

| Output | Artinya |
|--------|----------|
| `CUDA available: True` | ✅ GPU siap digunakan |
| `CUDA available: False` | ❌ Perlu install PyTorch versi CUDA |

> **VRAM minimum yang disarankan:** 4GB untuk YOLO11n/YOLOv8n dengan `imgsz=640`.

In [13]:
import torch

def check_gpu():
    print("=" * 50)
    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU             : {torch.cuda.get_device_name(0)}")
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM total      : {total_vram:.1f} GB")
        print("\n✅ GPU siap digunakan untuk training!")
    else:
        print("\n⚠️  WARNING: CUDA tidak tersedia — training akan berjalan di CPU (sangat lambat!)")
        print("   Solusi: Install PyTorch versi CUDA dari https://pytorch.org/get-started/locally/")

    print("=" * 50)

check_gpu()

PyTorch version : 2.11.0+cu126
CUDA available  : True
GPU             : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM total      : 6.4 GB

✅ GPU siap digunakan untuk training!


## Step 3 — Konfigurasi Path & Model

Sesuaikan path di bawah dengan struktur folder-mu.

### Struktur folder yang diharapkan:
```
D:\PrivAI\
├── datasets\
│   ├── data.yaml          ← file konfigurasi dataset
│   ├── images\
│   │   ├── train\         ← gambar training
│   │   └── val\           ← gambar validasi
│   └── labels\
│       ├── train\         ← label YOLO (.txt)
│       └── val\
└── runs\                  ← output training akan tersimpan di sini
```

### Contoh isi `data.yaml`:
```yaml
path: D:/PrivAI/datasets
train: images/train
val: images/val

nc: 5  # jumlah kelas
names: ['ktp', 'sim', 'paspor', 'npwp', 'kartu_kredit']
```

### Pilihan model base:
| Model | VRAM | Speed | Accuracy | Rekomendasi |
|-------|------|-------|----------|-------------|
| `yolo11n.pt` | ~2GB | ⚡⚡⚡ | ⭐⭐ | Edge device / HP |
| `yolov8n.pt` | ~2GB | ⚡⚡⚡ | ⭐⭐ | Alternatif v11n |
| `yolov8s.pt` | ~3GB | ⚡⚡ | ⭐⭐⭐ | Balance |
| `yolov8m.pt` | ~5GB | ⚡ | ⭐⭐⭐⭐ | Server |


In [14]:
from pathlib import Path

# ─────────────────────────────────────────────
#  ✏️  UBAH SESUAI PATH-MU
# ─────────────────────────────────────────────

DATA_YAML  = r"D:\PrivAI\datasets\data.yaml"  # path ke data.yaml
BASE_MODEL = "yolo11n.pt"                      # model base yang akan di-fine-tune
OUTPUT_DIR = r"D:\PrivAI\runs"                 # folder output hasil training
RUN_NAME   = "privai_v1"                       # nama run (subfolder di OUTPUT_DIR)

# ─────────────────────────────────────────────
#  Validasi path
# ─────────────────────────────────────────────

data_path = Path(DATA_YAML)
if data_path.exists():
    print(f"✅ data.yaml ditemukan : {DATA_YAML}")
else:
    print(f"❌ data.yaml TIDAK ditemukan: {DATA_YAML}")
    print("   Periksa kembali path DATA_YAML di atas!")

# Buat output dir jika belum ada
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"✅ Output dir siap    : {OUTPUT_DIR}")
print(f"✅ Model base         : {BASE_MODEL}")
print(f"✅ Nama run           : {RUN_NAME}")

✅ data.yaml ditemukan : D:\PrivAI\datasets\data.yaml
✅ Output dir siap    : D:\PrivAI\runs
✅ Model base         : yolo11n.pt
✅ Nama run           : privai_v1


## Step 4 — Hitung Batch Size Otomatis

Batch size terlalu besar → **OOM (Out of Memory)**. Terlalu kecil → training lambat.

Fungsi di bawah otomatis memilih batch size aman berdasarkan VRAM GPU-mu.

| VRAM | Batch Size | Catatan |
|------|-----------|----------|
| < 5GB | 8 | RTX 3050, GTX 1650, dll |
| 5–7GB | 16 | RTX 3060, GTX 1070 |
| 7–10GB | 24 | RTX 3070, RTX 2080 |
| ≥ 10GB | 32 | RTX 3080/4080/A100 |

> Kamu bisa override nilai ini secara manual dengan mengubah variabel `BATCH_SIZE` di sel berikutnya.

In [15]:
def auto_batch_size():
    """
    Estimasi batch size aman untuk YOLO11n/YOLOv8n dengan imgsz=640.
    Adjust jika kamu pakai model yang lebih besar (m/l/x).
    """
    if not torch.cuda.is_available():
        print("⚠️  Tidak ada GPU — menggunakan batch=4 (CPU fallback)")
        return 4

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

    if vram_gb < 5:
        batch = 8
    elif vram_gb < 7:
        batch = 16
    elif vram_gb < 10:
        batch = 24
    else:
        batch = 32

    print(f"VRAM terdeteksi : {vram_gb:.1f} GB")
    print(f"Batch size auto : {batch}")
    return batch


BATCH_SIZE = auto_batch_size()

# ── Override manual (uncomment jika diperlukan) ──
# BATCH_SIZE = 16

print(f"\n→ Batch size yang akan digunakan: {BATCH_SIZE}")

VRAM terdeteksi : 6.4 GB
Batch size auto : 16

→ Batch size yang akan digunakan: 16


## Step 5 — Training

Proses training dimulai di sini. Beberapa hal yang perlu diketahui:

### ⏱️ Estimasi durasi training (150 epoch, GPU mid-range):
| Dataset size | RTX 3060 | RTX 3080 |
|-------------|----------|----------|
| ~500 gambar | ~30 menit | ~15 menit |
| ~2000 gambar | ~2 jam | ~1 jam |
| ~10000 gambar | ~8 jam | ~4 jam |

### 📐 Augmentasi yang digunakan (dioptimalkan untuk dokumen):
| Parameter | Nilai | Alasan |
|-----------|-------|--------|
| `fliplr=0.0` | OFF | Teks jadi terbalik jika di-flip |
| `flipud=0.0` | OFF | Dokumen tidak pernah terbalik |
| `degrees=15` | ±15° | Simulasi dokumen diletakkan miring |
| `mosaic=1.0` | ON | Membantu deteksi objek kecil |
| `mixup=0.0` | OFF | Tidak cocok untuk dokumen berlabel |
| `hsv_v=0.4` | Aktif | Simulasi bayangan & pencahayaan berbeda |

### 🛑 Early Stopping:
Training otomatis berhenti jika mAP tidak meningkat selama `patience=30` epoch berturut-turut — menghemat waktu & mencegah overfitting.

> **Monitoring:** Buka **TensorBoard** di terminal untuk melihat grafik loss secara live:
> ```bash
> tensorboard --logdir D:\PrivAI\runs
> ```

In [16]:
from ultralytics import YOLO

# ── Load model base (otomatis download jika belum ada) ──
model = YOLO(BASE_MODEL)
device = 0 if torch.cuda.is_available() else "cpu"

print(f"Model  : {BASE_MODEL}")
print(f"Data   : {DATA_YAML}")
print(f"Device : {'GPU ' + torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")
print(f"Batch  : {BATCH_SIZE}")
print(f"Output : {OUTPUT_DIR}\\{RUN_NAME}")
print("\n🚀 Memulai training...\n")

results = model.train(
    data    = DATA_YAML,
    epochs  = 50,
    imgsz   = 640,
    batch   = BATCH_SIZE,
    device  = device,
    patience= 30,          # early stopping: berhenti jika tidak ada perbaikan 30 epoch
    save    = True,
    workers = 0,           # set 0 untuk Windows (hindari multiprocessing error)

    # ── Augmentasi dokumen ──
    degrees     = 15.0,    # rotasi ±15°
    shear       = 5.0,     # distorsi sudut
    perspective = 0.0005,  # perspektif 3D ringan
    scale       = 0.3,     # zoom in/out ±30%
    mosaic      = 1.0,     # gabung 4 gambar per batch
    mixup       = 0.0,     # dimatikan untuk dokumen

    # ── JANGAN flip teks ──
    fliplr      = 0.0,     # no horizontal flip
    flipud      = 0.0,     # no vertical flip

    # ── Variasi warna & cahaya ──
    hsv_h       = 0.015,
    hsv_s       = 0.5,
    hsv_v       = 0.4,

    # ── Output ──
    project     = OUTPUT_DIR,
    name        = RUN_NAME,
    plots       = True,    # simpan grafik training
    exist_ok    = True,    # overwrite jika nama run sudah ada
)

print("\n✅ Training selesai!")
print(f"   Hasil  : {OUTPUT_DIR}\\{RUN_NAME}")
print(f"   Best   : {OUTPUT_DIR}\\{RUN_NAME}\\weights\\best.pt")
print(f"   Last   : {OUTPUT_DIR}\\{RUN_NAME}\\weights\\last.pt")

Model  : yolo11n.pt
Data   : D:\PrivAI\datasets\data.yaml
Device : GPU NVIDIA GeForce RTX 4050 Laptop GPU
Batch  : 16
Output : D:\PrivAI\runs\privai_v1

🚀 Memulai training...

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.25  Python-3.12.6 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\PrivAI\datasets\data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.

## Step 6 — Validasi Model

Setelah training, evaluasi performa model menggunakan data validasi.

### Metrik yang dihasilkan:
| Metrik | Keterangan | Target |
|--------|------------|--------|
| **mAP50** | mAP pada IoU threshold 50% | > 0.85 |
| **mAP50-95** | mAP rata-rata IoU 50–95% | > 0.60 |
| **Precision** | Seberapa jarang false positive | > 0.85 |
| **Recall** | Seberapa jarang false negative | > 0.80 |

> **mAP50 > 0.9** = model sudah sangat baik untuk aplikasi privasi dokumen.


## 📁 Output yang dihasilkan setelah training:
```
D:\PrivAI\runs\privai_v1\
├── weights\
│   ├── best.pt          ← model terbaik (gunakan ini)
│   └── last.pt          ← model epoch terakhir
├── results.csv          ← log loss & mAP per epoch
├── confusion_matrix.png ← confusion matrix
├── results.png          ← grafik training
└── val_batch*.jpg       ← sample prediksi validasi
```